In [6]:
import re, time, numpy as np, pandas as pd, torch
from openai import OpenAI
from m1_utils import *
from m1_hyde import generate, MODEL, PV
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()                       # reads OPENAI_API_KEY from the environment
MODEL = "gpt-4o-mini"                   # <- change to a model your key can use; keep fixed for C3 and C4
print("model:", MODEL, "| prompt version:", PV)

eval_q = pd.read_parquet(OUT / "eval_query_order.parquet")
test = eval_q[eval_q["role"] == "eval_test"]
print("test queries:", len(test))
prose = generate("prose", "eval_test", test)
spec = generate("spec", "eval_test", test)
print(len(prose), len(spec))

model: gpt-4o-mini | prompt version: v1
test queries: 1295
prose cached 0 | todo 1295
spec cached 0 | todo 1295
1295 1295


In [7]:
from sentence_transformers import SentenceTransformer
queries, judg, corpus = load_all(); qrels = build_qrels(judg); pids = corpus["product_id"].to_numpy()
doc_emb = np.load(OUT / "doc_emb_minilm.npy")
bm25_all = np.load(OUT / "bm25_scores.npy", mmap_mode="r")
P = pd.read_parquet(OUT / "clf_probs_eval.parquet"); assert (P["query_id"].values == eval_q["query_id"].values).all()

rows = test.index.to_numpy(); qids = test["query_id"].tolist(); n = len(qids)
pE = P.loc[rows, "pE_T"].values
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu"); st.max_seq_length = 256
enc = lambda d: st.encode([d[q] for q in qids], batch_size=64, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
qr = np.load(OUT / "query_emb_raw_minilm.npy")[rows]; qp, qs = enc(prose), enc(spec)
VEC = {"raw": qr, "prose": qp, "spec": qs, "prose_q": (qr + qp) / 2, "spec_q": (qr + qs) / 2}   # avg of embeddings == avg of scores

K = 100
def top10(s):
    t = np.argpartition(-s, 10)[:10]; return list(pids[t[np.argsort(-s[t])]])
def run(vec=None, alpha=None):
    ranked = {}
    for j, q in enumerate(qids):
        b = np.asarray(bm25_all[rows[j]])
        if vec is None: ranked[q] = top10(b); continue
        d = VEC[vec][j] @ doc_emb.T
        if alpha is None: ranked[q] = top10(d); continue
        cand = np.union1d(np.argpartition(-b, K)[:K], np.argpartition(-d, K)[:K])
        s = alpha[j] * minmax(b[cand]) + (1 - alpha[j]) * minmax(d[cand])
        ranked[q] = list(pids[cand[np.argsort(-s)[:10]]])
    return evaluate(ranked, qrels)

fix = np.full(n, 0.4)
COND = {  # literal proposal table
    "C1 BM25 only": (None, None), "C2 raw dense": ("raw", None),
    "C3 prose HyDE (original, dense only)": ("prose", None),
    "C4 PROPOSED: spec HyDE + alpha=P(Exact)": ("spec", pE),
    # extended
    "X1 raw dense + BM25 (a=0.4)": ("raw", fix),
    "X2 prose+query HyDE, dense only": ("prose_q", None), "X3 spec+query HyDE, dense only": ("spec_q", None),
    "X4 prose+query HyDE + BM25 (a=0.4)": ("prose_q", fix), "X5 spec+query HyDE + BM25 (a=0.4)": ("spec_q", fix),
    "X6 spec+query HyDE + BM25 (a=P(Exact))": ("spec_q", pE),
    "X7 spec HyDE + BM25 (a=0.4)": ("spec", fix), "X8 prose HyDE + BM25 (a=0.4)": ("prose", fix)}
res = {k: run(*v) for k, v in COND.items()}

sm = (test["small"].values == 1)
tab = pd.DataFrame({k: {"NDCG@10": r["ndcg@10"].mean(), "MRR@10": r["mrr@10"].mean(),
                        "NDCG@10 small": r["ndcg@10"].values[sm].mean(), "MRR@10 small": r["mrr@10"].values[sm].mean()}
                    for k, r in res.items()}).T.round(4)
print(f"TEST: n={n} (small_version n={sm.sum()})\n"); print(tab.to_string())

def boot(a, b, nb=5000, seed=0):
    d = a - b; r = np.random.default_rng(seed)
    m = np.array([d[r.integers(0, len(d), len(d))].mean() for _ in range(nb)]); return d.mean(), np.percentile(m, [2.5, 97.5])
CONTR = [("HyDE gain: X4 prose+q fused - X1 raw fused", "X4 prose+query HyDE + BM25 (a=0.4)", "X1 raw dense + BM25 (a=0.4)"),
         ("Novelty 1 (avg-query): X5 spec - X4 prose", "X5 spec+query HyDE + BM25 (a=0.4)", "X4 prose+query HyDE + BM25 (a=0.4)"),
         ("Novelty 1 (pure): X7 spec - X8 prose, fused", "X7 spec HyDE + BM25 (a=0.4)", "X8 prose HyDE + BM25 (a=0.4)"),
         ("Novelty 1 (pure, dense only): spec - prose", None, None),
         ("Novelty 2: X6 alpha=P(Exact) - X5 fixed 0.4", "X6 spec+query HyDE + BM25 (a=P(Exact))", "X5 spec+query HyDE + BM25 (a=0.4)"),
         ("Novelty 2 literal: C4 - X7 (fixed 0.4)", "C4 PROPOSED: spec HyDE + alpha=P(Exact)", "X7 spec HyDE + BM25 (a=0.4)"),
         ("C4 proposed - C1 BM25", "C4 PROPOSED: spec HyDE + alpha=P(Exact)", "C1 BM25 only"),
         ("C4 proposed - C3 original HyDE", "C4 PROPOSED: spec HyDE + alpha=P(Exact)", "C3 prose HyDE (original, dense only)"),
         ("X5 best fixed - C1 BM25", "X5 spec+query HyDE + BM25 (a=0.4)", "C1 BM25 only")]
spec_dense = run("spec", None)
print("\nPaired bootstrap, NDCG@10 (95% CI):")
for name, a, b in CONTR:
    x = spec_dense["ndcg@10"].values if a is None else res[a]["ndcg@10"].values
    y = res["C3 prose HyDE (original, dense only)"]["ndcg@10"].values if b is None else res[b]["ndcg@10"].values
    m, ci = boot(x, y); print(f"{name:55s} {m:+.4f}  [{ci[0]:+.4f}, {ci[1]:+.4f}]")

tab.to_csv(OUT / "final_test_results_v1.csv")
pd.concat({k: r for k, r in res.items()}, axis=1).to_parquet(OUT / "final_test_per_query_v1.parquet")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

TEST: n=1295 (small_version n=507)

                                         NDCG@10  MRR@10  NDCG@10 small  MRR@10 small
C1 BM25 only                              0.2546  0.3780         0.2008        0.3103
C2 raw dense                              0.2561  0.3974         0.2099        0.3243
C3 prose HyDE (original, dense only)      0.2496  0.3910         0.2055        0.3287
C4 PROPOSED: spec HyDE + alpha=P(Exact)   0.2333  0.3583         0.1805        0.2867
X1 raw dense + BM25 (a=0.4)               0.3038  0.4443         0.2498        0.3792
X2 prose+query HyDE, dense only           0.2764  0.4185         0.2321        0.3638
X3 spec+query HyDE, dense only            0.2688  0.4158         0.2206        0.3480
X4 prose+query HyDE + BM25 (a=0.4)        0.3220  0.4631         0.2693        0.3952
X5 spec+query HyDE + BM25 (a=0.4)         0.3223  0.4730         0.2661        0.4147
X6 spec+query HyDE + BM25 (a=P(Exact))    0.2896  0.4353         0.2326        0.3649
X7 spec HyDE + BM2

In [8]:
from rouge_score import rouge_scorer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification

doc_of = dict(zip(corpus["product_id"], corpus["doc_text"]))
exact = judg[judg["esci_label"] == "E"].groupby("query_id")["product_id"].apply(list)
sc = rouge_scorer.RougeScorer(["rougeL"])
def rl(gen):
    f, r = [], []
    for q in qids:
        s = [sc.score(doc_of[p], gen[q])["rougeL"] for p in exact[q][:5]]
        f.append(np.mean([x.fmeasure for x in s])); r.append(np.mean([x.recall for x in s]))
    return np.array(f), np.array(r)
(fp, rp), (fs, rs) = rl(prose), rl(spec)
print(f"ROUGE-L test: prose F1 {fp.mean():.3f} recall {rp.mean():.3f} | spec F1 {fs.mean():.3f} recall {rs.mean():.3f}")
print("avg words: prose %.0f | spec %.0f" % (np.mean([len(prose[q].split()) for q in qids]), np.mean([len(spec[q].split()) for q in qids])))
m, ci = boot(fs, fp); print(f"F1 spec-prose {m:+.4f} [{ci[0]:+.4f}, {ci[1]:+.4f}]")

LAB = ["Exact", "Hybrid", "Semantic"]
tok = AutoTokenizer.from_pretrained(OUT / "clf_distilbert"); mdl = AutoModelForSequenceClassification.from_pretrained(OUT / "clf_distilbert").eval()
cte = pd.read_csv(OUT / "classifier_test.csv")
with torch.no_grad():
    e = tok(cte["query"].tolist(), padding=True, truncation=True, max_length=64, return_tensors="pt")
    pr = mdl(**e).logits.argmax(1).numpy()
y = cte["intent"].map({l: i for i, l in enumerate(LAB)}).values
acc = accuracy_score(y, pr); nn = len(y); z = 1.96
c = (acc + z*z/(2*nn)) / (1 + z*z/nn); h = z*np.sqrt(acc*(1-acc)/nn + z*z/(4*nn*nn)) / (1 + z*z/nn)
print(f"\nCLASSIFIER TEST (n={nn}): accuracy {acc:.3f} (Wilson 95% [{c-h:.3f}, {c+h:.3f}]) | macro-F1 {f1_score(y, pr, average='macro'):.3f}")
print(classification_report(y, pr, target_names=LAB, digits=3)); print(confusion_matrix(y, pr))

ROUGE-L test: prose F1 0.115 recall 0.144 | spec F1 0.117 recall 0.079
avg words: prose 123 | spec 22
F1 spec-prose +0.0018 [-0.0009, +0.0046]


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


CLASSIFIER TEST (n=83): accuracy 0.747 (Wilson 95% [0.644, 0.828]) | macro-F1 0.753
              precision    recall  f1-score   support

       Exact      0.800     0.800     0.800        20
      Hybrid      0.727     0.750     0.738        32
    Semantic      0.733     0.710     0.721        31

    accuracy                          0.747        83
   macro avg      0.754     0.753     0.753        83
weighted avg      0.747     0.747     0.747        83

[[16  3  1]
 [ 1 24  7]
 [ 3  6 22]]


In [9]:
for name, a, b in [("MRR: X5 spec+q fused - X4 prose+q fused", "X5 spec+query HyDE + BM25 (a=0.4)", "X4 prose+query HyDE + BM25 (a=0.4)"),
                   ("MRR: X4 prose+q fused - X1 raw fused", "X4 prose+query HyDE + BM25 (a=0.4)", "X1 raw dense + BM25 (a=0.4)"),
                   ("MRR: X5 spec+q fused - X1 raw fused", "X5 spec+query HyDE + BM25 (a=0.4)", "X1 raw dense + BM25 (a=0.4)")]:
    m, ci = boot(res[a]["mrr@10"].values, res[b]["mrr@10"].values)
    print(f"{name:45s} {m:+.4f} [{ci[0]:+.4f}, {ci[1]:+.4f}]")

MRR: X5 spec+q fused - X4 prose+q fused       +0.0099 [-0.0031, +0.0226]
MRR: X4 prose+q fused - X1 raw fused          +0.0188 [+0.0075, +0.0303]
MRR: X5 spec+q fused - X1 raw fused           +0.0287 [+0.0151, +0.0417]
